# 03 - Diffusion Models: Concepts (OPTIONAL)

## Learning Objectives
- Understand the forward (noising) and reverse (denoising) diffusion processes
- Visualize noise schedules
- Know the high-level architecture of DDPM

## Prerequisites
- DL100-DL200 (neural networks, training loops)
- Basic probability

## Table of Contents
1. [Diffusion Intuition](#1)
2. [Forward Process](#2)
3. [Reverse Process](#3)
4. [Noise Schedule Visualization](#4)
5. [Simple 1D Denoising Demo](#5)
6. [Modern Diffusion Models](#6)
7. [Exercises](#7)
8. [Common Mistakes & Debugging Tips](#8)

> **Note:** This notebook is conceptual. No heavy training required.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import sys; sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed
set_global_seed(42)

<a id='1'></a>
## 1. Diffusion Intuition

Imagine dropping ink into water:
- **Forward process:** The ink gradually diffuses until it's uniformly spread (pure noise)
- **Reverse process:** If we could reverse time, we'd recover the original ink pattern

Diffusion models learn to reverse the noising process:
1. **Forward:** Gradually add Gaussian noise to data over $T$ timesteps
2. **Reverse:** Train a neural network to predict and remove the noise, step by step

```
Clean Image → [+noise] → [+noise] → ... → [+noise] → Pure Noise
     x_0         x_1        x_2               x_T       ≈ N(0,1)
              ←──────── Neural Net learns to reverse ────────→
```

<a id='2'></a>
## 2. Forward Process (Adding Noise)

At each timestep $t$, we add a small amount of Gaussian noise:

$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1-\beta_t}\, x_{t-1},\; \beta_t \mathbf{I})$$

where $\beta_t$ is the noise schedule (small values, e.g., 0.0001 to 0.02).

With the reparameterization trick, we can jump directly to any timestep:

$$x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \epsilon, \quad \epsilon \sim \mathcal{N}(0, \mathbf{I})$$

where $\alpha_t = 1 - \beta_t$ and $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$.

In [ ]:
# Demonstrate forward diffusion on a simple 2D dataset
def make_spiral(n=1000):
    t = torch.linspace(0, 4 * np.pi, n)
    x = t * torch.cos(t) / (4 * np.pi)
    y = t * torch.sin(t) / (4 * np.pi)
    return torch.stack([x, y], dim=1)

data = make_spiral(2000)

# Linear noise schedule
T = 100
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0)

def forward_diffusion(x0, t, alpha_bar):
    """Add noise to x0 at timestep t."""
    noise = torch.randn_like(x0)
    sqrt_ab = torch.sqrt(alpha_bar[t]).unsqueeze(-1)
    sqrt_one_minus_ab = torch.sqrt(1 - alpha_bar[t]).unsqueeze(-1)
    return sqrt_ab * x0 + sqrt_one_minus_ab * noise, noise

# Visualize progressive noising
fig, axes = plt.subplots(1, 5, figsize=(18, 3))
timesteps = [0, 10, 30, 60, 99]
for ax, t in zip(axes, timesteps):
    t_tensor = torch.full((len(data),), t, dtype=torch.long)
    noisy, _ = forward_diffusion(data, t_tensor, alpha_bar)
    ax.scatter(noisy[:, 0], noisy[:, 1], s=1, alpha=0.5)
    ax.set_title(f't = {t}')
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
plt.suptitle('Forward Diffusion: Spiral → Noise', fontsize=13)
plt.tight_layout(); plt.show()

<a id='3'></a>
## 3. Reverse Process (Denoising)

A neural network $\epsilon_\theta(x_t, t)$ is trained to predict the noise $\epsilon$ that was added:

$$L = \mathbb{E}_{t, x_0, \epsilon}\left[\|\epsilon - \epsilon_\theta(x_t, t)\|^2\right]$$

At inference, we start from pure noise and iteratively denoise:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\epsilon_\theta(x_t, t)\right) + \sigma_t z$$

<a id='4'></a>
## 4. Noise Schedule Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(betas.numpy())
axes[0].set_title('β_t (noise added per step)')
axes[0].set_xlabel('Timestep')

axes[1].plot(alpha_bar.numpy())
axes[1].set_title('ᾱ_t (cumulative signal remaining)')
axes[1].set_xlabel('Timestep')

axes[2].plot(torch.sqrt(alpha_bar).numpy(), label='√ᾱ (signal)')
axes[2].plot(torch.sqrt(1 - alpha_bar).numpy(), label='√(1-ᾱ) (noise)')
axes[2].set_title('Signal vs Noise over time')
axes[2].set_xlabel('Timestep')
axes[2].legend()

for ax in axes: ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

<a id='5'></a>
## 5. Simple 1D Denoising Demo

Train a tiny network to denoise 1D Gaussian data as a proof of concept.

In [ ]:
# Simple denoiser: given noisy 1D data + timestep, predict the noise
class SimpleDenoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 128), nn.ReLU(),  # input: [noisy_x, timestep_embedding]
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 1)
        )
    def forward(self, x_noisy, t_normalized):
        inp = torch.cat([x_noisy, t_normalized], dim=-1)
        return self.net(inp)

# Train to denoise 1D data from a mixture of Gaussians
denoiser = SimpleDenoiser()
optimizer = torch.optim.Adam(denoiser.parameters(), lr=1e-3)

T_small = 50
betas_s = torch.linspace(1e-4, 0.02, T_small)
alphas_s = 1.0 - betas_s
alpha_bar_s = torch.cumprod(alphas_s, dim=0)

for step in range(3000):
    # Sample real data: mixture of two Gaussians
    x0 = torch.cat([torch.randn(64, 1) - 2, torch.randn(64, 1) + 2])
    
    # Sample random timesteps
    t = torch.randint(0, T_small, (128,))
    noise = torch.randn_like(x0)
    
    sqrt_ab = torch.sqrt(alpha_bar_s[t]).unsqueeze(-1)
    sqrt_1_ab = torch.sqrt(1 - alpha_bar_s[t]).unsqueeze(-1)
    x_noisy = sqrt_ab * x0 + sqrt_1_ab * noise
    
    t_norm = (t.float() / T_small).unsqueeze(-1)
    pred_noise = denoiser(x_noisy, t_norm)
    
    loss = nn.MSELoss()(pred_noise, noise)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

print(f"Final denoising loss: {loss.item():.4f}")

<a id='6'></a>
## 6. Modern Diffusion Models

- **DDPM** (2020): Denoising Diffusion Probabilistic Models — foundational work
- **DDIM** (2021): Faster sampling with fewer steps
- **Stable Diffusion** (2022): Diffusion in latent space (faster, lower memory)
- **DALL-E 2/3, Midjourney, Imagen**: Text-to-image using diffusion + text encoders

These models use **U-Net** architectures with attention layers as the denoiser $\epsilon_\theta$.

<a id='7'></a>
## 7. Exercises

**Exercise 1:** Try a cosine noise schedule instead of linear: $\bar{\alpha}_t = \cos^2\left(\frac{t/T + s}{1+s} \cdot \frac{\pi}{2}\right)$. Plot it.

**Exercise 2:** Modify the forward diffusion to work on a 2D spiral dataset and visualize.

**Exercise 3:** What happens if you use too many or too few diffusion steps?

<a id='8'></a>
## 8. Common Mistakes & Debugging Tips

| Mistake | Fix |
|---------|-----|
| Too few timesteps | Poor quality; need enough steps for gradual denoising |
| Wrong noise schedule | Linear is OK for start; cosine often better |
| Not conditioning on timestep | Model needs to know "how noisy" the input is |
| Training too briefly | Diffusion models need many iterations to converge |